# Capstone companion --- Chapter 7: Cost, Latency and Budgets

An agent that reasons in a loop will, if left unbounded, consume tokens, wall-clock time, tool calls and dollars without limit; a run that never terminates is a failure of resource discipline before it is a failure of correctness. Chapter~7 addresses this by giving every run an explicit budget and a tracker that accumulates consumption against it, so the loop terminates when any axis is exhausted. This companion reads that principle on the capstone banking complaint agent.

The capstone represents a budget as two objects in `agentlab.core`. A `Budget` declares per-axis upper bounds; a `BudgetTracker` accumulates a `Consumption` against one budget and reports whether any axis is spent. The run loop consults the tracker before each step, so the concern is enforced at the loop boundary rather than inside any single tool.

In [ ]:
from forgeloop.agents.core import Budget, BudgetTracker, Consumption
import inspect

print('Budget fields      :', list(Budget.__dataclass_fields__))
print('Consumption fields :', list(Consumption.__dataclass_fields__))
print('BudgetTracker API  :',
      [m for m in dir(BudgetTracker) if not m.startswith('_')])

## Declaring a budget for a complaint run

A `Budget` caps four axes: `tokens`, `seconds`, `tool_calls` and `dollars`. A value of `None` leaves an axis unbounded. A single complaint passes through five tools --- classify, extract, search, flag and draft --- so a healthy run makes on the order of five tool calls. The budget below sets a tool-call ceiling that admits that path with a small margin, together with token and dollar ceilings sized for one short response.

In [ ]:
budget = Budget(tokens=4000, seconds=30.0, tool_calls=6, dollars=0.05)
budget

## Tracking consumption across the run

A `BudgetTracker` is constructed against one budget and starts a wall-clock timer. As the run proceeds the tracker records what each step consumes: `record_tool_call` on every tool invocation, `record_tokens` and `record_dollars` on every model call. The simulation below records the five tool calls of a nominal complaint run together with their token and dollar cost, then reads back the accumulated consumption.

In [ ]:
tracker = BudgetTracker(budget)

# A nominal five-tool complaint run: (tool name, tokens, dollars).
steps = [
    ('classify_complaint', 320, 0.0032),
    ('extract_facts',      540, 0.0054),
    ('search_policy',      410, 0.0041),
    ('flag_regulatory',    280, 0.0028),
    ('draft_response',     900, 0.0090),
]
for name, toks, cost in steps:
    tracker.record_tool_call()
    tracker.record_tokens(toks)
    tracker.record_dollars(cost)

c = tracker.consumption()
print(f'tool calls : {c.tool_calls}/{budget.tool_calls}')
print(f'tokens     : {c.tokens}/{budget.tokens}')
print(f'dollars    : {c.dollars:.4f}/{budget.dollars:.4f}')
print(f'elapsed    : {c.seconds:.4f}s (budget {budget.seconds}s)')
print('exhausted  :', tracker.exhausted())

The nominal run stays inside every axis, so `exhausted` is false and the loop would continue to a normal `Finish`. The tracker reports elapsed seconds from its own timer rather than from a recorded value, because wall-clock time accrues whether or not a step chooses to spend it.

## Reaching the ceiling

The purpose of the tracker is to name the axis that terminates a run. A complaint that loops --- re-searching policy after each partial answer --- spends tool calls without converging. `reason_exhausted` returns a human-readable string identifying the first axis at or above its bound, or `None` while the budget holds. The loop below records tool calls until the tracker reports exhaustion.

In [ ]:
looping = BudgetTracker(Budget(tool_calls=6))
for i in range(1, 11):
    looping.record_tool_call()
    reason = looping.reason_exhausted()
    print(f'after call {i:2d}: exhausted={looping.exhausted()!s:5s}  reason={reason}')
    if reason is not None:
        break

## What the run loop does at the ceiling

The capstone run loop in `agentlab.core.loop` consults the tracker at the top of every step. When `exhausted` becomes true it does not silently stop: it synthesizes a terminal `Escalate` action whose reason is the string from `reason_exhausted` and whose context marks the source as the budget, then sets the run status to `failed`. A run that overspends therefore ends in a recorded, attributable escalation rather than an open-ended consumption of resources.

In [ ]:
from forgeloop.agents.core import Escalate

# Reconstruct the terminal record the loop emits when the budget is spent.
reason = looping.reason_exhausted()
terminal = Escalate(reason=reason, context={'source': 'budget'})
print('action  :', terminal.kind)
print('reason  :', terminal.reason)
print('source  :', terminal.context['source'])

This is the capstone's realization of Chapter~7: a complaint run carries an explicit `Budget`, a `BudgetTracker` accumulates consumption across its tools, and the loop terminates the run with an attributable budget escalation the moment any axis is exhausted. Chapter~16 assembles the five tools into the governed workflow, where the budget tracker bounds the same loop that the earlier chapters typed, guarded and sequenced.